## 1) 导入依赖

In [1]:
# 如环境已安装可注释掉本单元
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q opencv-python numpy trimesh ipywidgets

import os
import sys
import glob
import gc
import time
import tempfile
import re
from pathlib import Path

import cv2
import torch
import numpy as np
import trimesh

from IPython.display import display, HTML, FileLink

# 关键：保证可导入 vggt 代码
PROJECT_ROOT = Path('/home/viplab/liukejia/project/vggt').resolve()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'vggt'))
sys.path.append(str(PROJECT_ROOT.parent / 'reconstruct'))

from utils.visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
from utils.camera import parse_json_camera_extrinsic_intrinsic

print('PROJECT_ROOT =', PROJECT_ROOT)

/home/viplab/anaconda3/envs/vggt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT = /home/viplab/liukejia/project/vggt


## 2) 参数配置

In [2]:
# ===== 必填参数 =====
input_path = '/home/viplab/liukejia/project/result/inference_images'
output_path = '/home/viplab/liukejia/project/result/reconstruct_pred_scale3.1/'

# ===== 处理参数 =====
force_resize = True
conf_thres = 50.0
prediction_mode = 'Depthmap and Camera Branch'
save_npz = False

# ===== 相机参数来源配置 =====
use_camconf_camera = True
camconf_dir = '/data/test_grid4_from_test_grid4/camconf'
camconf_angle_order = (0, 90, 180, 270)
strict_camconf = True
invert_camera_direction = False
debug_print_camera = True

# ===== 只做刚体/相似变换（保几何） =====
# none | rigid | similarity | camera_fit
post_align_mode = 'camera_fit'
post_align_scale = 1.0

# ===== camera_fit 额外缩放：让预测点范围贴近文件相机范围 =====
camera_fit_match_range = True
camera_fit_range_gain = 1.0
camera_fit_scale_min = 0.05
camera_fit_scale_max = 20.0
camera_fit_range_apply_to_camera = True

valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('input_path =', input_path)
print('output_path =', output_path)
print('use_camconf_camera =', use_camconf_camera)
print('camconf_dir =', camconf_dir)
print('camconf_angle_order =', camconf_angle_order)
print('strict_camconf =', strict_camconf)
print('invert_camera_direction =', invert_camera_direction)
print('debug_print_camera =', debug_print_camera)
print('post_align_mode =', post_align_mode)
print('post_align_scale =', post_align_scale)
print('camera_fit_match_range =', camera_fit_match_range)
print('camera_fit_range_gain =', camera_fit_range_gain)
print('camera_fit_scale_min/max =', (camera_fit_scale_min, camera_fit_scale_max))
print('camera_fit_range_apply_to_camera =', camera_fit_range_apply_to_camera)
print('save_npz =', save_npz)

input_path = /home/viplab/liukejia/project/result/inference_images
output_path = /home/viplab/liukejia/project/result/reconstruct_pred_scale3.1/
use_camconf_camera = True
camconf_dir = /data/test_grid4_from_test_grid4/camconf
camconf_angle_order = (0, 90, 180, 270)
strict_camconf = True
invert_camera_direction = False
debug_print_camera = True
post_align_mode = camera_fit
post_align_scale = 1.0
camera_fit_match_range = True
camera_fit_range_gain = 1.0
camera_fit_scale_min/max = (0.05, 20.0)
camera_fit_range_apply_to_camera = True
save_npz = False


## 3) 加载 VGGT 模型

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

print('Initializing VGGT...')
model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    amp_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    amp_dtype = None

print('amp_dtype =', amp_dtype)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = cuda
Initializing VGGT...


amp_dtype = torch.bfloat16


## 4) I/O 与切图工具

In [4]:
def collect_input_images(input_path: str, valid_exts=None):
    if valid_exts is None:
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f'input_path not found: {p}')

    if p.is_file():
        if p.suffix.lower() not in valid_exts:
            raise ValueError(f'Unsupported file extension: {p.suffix}')
        return [p]

    if p.is_dir():
        files = [x for x in sorted(p.iterdir()) if x.is_file() and x.suffix.lower() in valid_exts]
        if len(files) == 0:
            raise ValueError(f'No image files found in folder: {p}')
        return files

    raise ValueError(f'Invalid input_path: {p}')


def read_single_image_512(image_path: str, force_resize: bool = True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path.name}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'{image_path.name}: expect 512x512, got {(w, h)}')

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb


def split_512_to_4x256_and_save(img_rgb: np.ndarray, target_dir: str):
    target_dir = Path(target_dir)
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    patches = [
        img_rgb[0:256, 0:256],
        img_rgb[0:256, 256:512],
        img_rgb[256:512, 0:256],
        img_rgb[256:512, 256:512],
    ]

    patch_paths = []
    for i, p in enumerate(patches):
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(p, cv2.COLOR_RGB2BGR))
        patch_paths.append(str(out_path))

    cv2.imwrite(str(target_dir / 'original_512.png'), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return patch_paths


input_images = collect_input_images(input_path, valid_exts=valid_exts)
print(f'Found {len(input_images)} image(s).')
print('First 5:', [p.name for p in input_images[:5]])

Found 9 image(s).
First 5: ['num123_-800_100_0_0007.png', 'num123_-800_150_0_0008.png', 'num123_-800_200_0_0009.png', 'num123_-850_100_0_0004.png', 'num123_-850_150_0_0005.png']


## 5) 相机/几何辅助

In [5]:
def extract_camconf_base_stem(image_stem: str) -> str:
    m = re.match(r'^num123_([^_]+_[^_]+_[^_]+)_[^_]+$', image_stem)
    if m:
        return m.group(1)
    parts = image_stem.split('_')
    if len(parts) >= 5 and parts[0] == 'num123':
        return '_'.join(parts[1:4])
    if len(parts) >= 3:
        return '_'.join(parts[:3])
    raise ValueError(f'Cannot parse camconf stem from image stem: {image_stem}')


def _convert_extrinsic_shape_like(src_extr: np.ndarray, ref_extr: np.ndarray) -> np.ndarray:
    src = np.asarray(src_extr, dtype=np.float32)
    ref = np.asarray(ref_extr)
    if src.ndim != 3 or ref.ndim != 3:
        raise ValueError(f'extrinsic ndim error: src={src.shape}, ref={ref.shape}')
    if ref.shape[1:] == (3, 4):
        if src.shape[1:] == (3, 4):
            return src
        if src.shape[1:] == (4, 4):
            return src[:, :3, :4]
        raise ValueError(f'unsupported src extrinsic shape: {src.shape}')
    if ref.shape[1:] == (4, 4):
        if src.shape[1:] == (4, 4):
            return src
        if src.shape[1:] == (3, 4):
            out = np.tile(np.eye(4, dtype=np.float32), (src.shape[0], 1, 1))
            out[:, :3, :4] = src
            return out
        raise ValueError(f'unsupported src extrinsic shape: {src.shape}')
    raise ValueError(f'unsupported ref extrinsic shape: {ref.shape}')


def _camera_centers_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    return c2w[:, :3, 3]


def _estimate_similarity_umeyama(src: np.ndarray, dst: np.ndarray, with_scale: bool = True):
    src = np.asarray(src, dtype=np.float32)
    dst = np.asarray(dst, dtype=np.float32)
    if src.shape != dst.shape or src.ndim != 2 or src.shape[1] != 3:
        raise ValueError(f'bad shapes: src={src.shape}, dst={dst.shape}')
    mu_s = src.mean(axis=0)
    mu_d = dst.mean(axis=0)
    X = src - mu_s[None, :]
    Y = dst - mu_d[None, :]
    cov = (Y.T @ X) / src.shape[0]
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3, dtype=np.float32)
    if np.linalg.det(U @ Vt) < 0:
        S[-1, -1] = -1.0
    R = (U @ S @ Vt).astype(np.float32)
    if with_scale:
        var_s = np.mean(np.sum(X * X, axis=1)) + 1e-8
        scale = float(np.sum(D * np.diag(S)) / var_s)
    else:
        scale = 1.0
    t = (mu_d - scale * (R @ mu_s)).astype(np.float32)
    return scale, R, t


def _apply_similarity_to_points_only(predictions: dict, s: float, R: np.ndarray, t: np.ndarray) -> dict:
    wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
    pts = wp.reshape(-1, 3)
    pts_new = (s * (pts @ R.T) + t[None, :]).astype(np.float32)
    predictions['world_points_from_depth'] = pts_new.reshape(wp.shape)
    return predictions


def _robust_span_diag(points: np.ndarray, q_low: float = 5.0, q_high: float = 95.0) -> float:
    p = np.asarray(points, dtype=np.float32)
    if p.ndim != 2 or p.shape[1] != 3 or p.shape[0] == 0:
        return 0.0
    lo = np.percentile(p, q_low, axis=0)
    hi = np.percentile(p, q_high, axis=0)
    return float(np.linalg.norm(hi - lo))


def _load_file_camera_centers_for_image(
    image_path: Path,
    camconf_dir: str,
    angle_order=(0, 90, 180, 270),
    H: int = 256,
    W: int = 256,
):
    base_stem = extract_camconf_base_stem(image_path.stem)
    centers = []
    valid_angles = []
    for a in angle_order:
        p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
        if not p.exists():
            continue
        ext44, _ = parse_json_camera_extrinsic_intrinsic(
            str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
        )
        c = _camera_centers_from_extr(ext44.squeeze(0).detach().cpu().numpy()[None, ...])[0]
        centers.append(c.astype(np.float32))
        valid_angles.append(int(a))
    if len(centers) == 0:
        raise FileNotFoundError(f'no camconf found for {image_path.name} in {camconf_dir}')
    return np.stack(centers, axis=0), valid_angles


def _load_camconf_all_cameras_for_image(
    image_path: Path,
    camconf_dir: str,
    angle_order=(0, 90, 180, 270),
    H: int = 256,
    W: int = 256,
):
    base_stem = extract_camconf_base_stem(image_path.stem)
    exts, intrs, paths, valid_angles = [], [], [], []
    for a in angle_order:
        p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
        if not p.exists():
            continue
        ext44, intr33 = parse_json_camera_extrinsic_intrinsic(
            str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
        )
        exts.append(ext44.squeeze(0).detach().cpu().numpy().astype(np.float32))
        intrs.append(intr33.squeeze(0).detach().cpu().numpy().astype(np.float32))
        paths.append(str(p))
        valid_angles.append(int(a))
    if len(exts) == 0:
        raise FileNotFoundError(f'no camconf found for {image_path.name} in {camconf_dir}')
    return np.stack(exts, axis=0), np.stack(intrs, axis=0), paths, valid_angles

## 6) run_model（仅保留一个版本）

In [6]:
def run_model(
    target_dir: str,
    model,
    image_path: Path = None,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
) -> dict:
    target_dir = Path(target_dir)
    image_names = sorted(glob.glob(str(target_dir / 'images' / '*')))
    if len(image_names) == 0:
        raise ValueError(f'No images found in {target_dir}')

    images = load_and_preprocess_images(image_names).to(device)

    with torch.no_grad():
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(dtype=amp_dtype):
                predictions = model(images)
        else:
            predictions = model(images)

    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
    predictions['extrinsic'] = extrinsic
    predictions['intrinsic'] = intrinsic

    for key in list(predictions.keys()):
        if isinstance(predictions[key], torch.Tensor):
            predictions[key] = predictions[key].detach().cpu().numpy().squeeze(0)

    predictions['pose_enc_list'] = None
    predictions['camera_source'] = 'prediction'

    depth_map = predictions['depth']
    predictions['world_points_from_depth'] = unproject_depth_map_to_point_map(
        depth_map, predictions['extrinsic'], predictions['intrinsic']
    )

    if not use_camconf_camera:
        return predictions

    if image_path is None or camconf_dir is None:
        raise ValueError('camconf 模式需要 image_path 与 camconf_dir')

    try:
        H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
        n_pred = int(predictions['depth'].shape[0])
        file_ext, file_intr, file_jsons, valid_angles = _load_camconf_all_cameras_for_image(
            image_path=image_path,
            camconf_dir=camconf_dir,
            angle_order=camconf_angle_order,
            H=H,
            W=W,
        )
        if file_ext.shape[0] < n_pred:
            raise ValueError(f'camconf views not enough: need={n_pred}, got={file_ext.shape[0]}')

        pred_extr_orig = np.asarray(predictions['extrinsic'], dtype=np.float32)
        pred_centers = _camera_centers_from_extr(pred_extr_orig)

        file_ext_like = _convert_extrinsic_shape_like(file_ext[:n_pred], pred_extr_orig)
        file_centers = _camera_centers_from_extr(np.asarray(file_ext_like, dtype=np.float32))

        n = min(len(pred_centers), len(file_centers))
        s, R, t = _estimate_similarity_umeyama(pred_centers[:n], file_centers[:n], with_scale=True)
        predictions = _apply_similarity_to_points_only(predictions, s=s, R=R, t=t)

        predictions['extrinsic'] = file_ext_like.astype(np.float32)
        predictions['intrinsic'] = file_intr[:n_pred].astype(np.float32)
        predictions['camera_source'] = 'camconf_exact_dense'
        predictions['camera_json_paths'] = np.array(file_jsons[:n_pred], dtype=object)

        pts = np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3)
        conf = np.asarray(predictions.get('depth_conf', np.ones((pts.shape[0],), dtype=np.float32))).reshape(-1)
        thr = np.percentile(conf, conf_thres) if conf.size else 0.0
        kept = int(np.sum((conf >= thr) & (conf > 1e-5))) if conf.size else int(pts.shape[0])
        print(
            f'[EXACT_FIX] {image_path.name} | mode=camconf_exact_dense | '
            f'similarity_scale={float(s):.4f} | points_kept@p{conf_thres}={kept}'
        )

    except Exception as e:
        if strict_camconf:
            raise
        print(f'[Warn][EXACT_FIX] failed, fallback prediction: {e}')

    return predictions

## 7) 导出与对齐

In [7]:
def export_npz_and_glb(
    sample_target_dir: str,
    predictions: dict,
    output_dir: str,
    output_stem: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    save_npz: bool = True,
):
    sample_target_dir = Path(sample_target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    glb_path = output_dir / f'{output_stem}.glb'
    npz_path = output_dir / f'{output_stem}.npz'

    if save_npz:
        np.savez(npz_path, **predictions)

    glbscene = predictions_to_glb(
        predictions,
        conf_thres=conf_thres,
        filter_by_frames='All',
        show_cam=True,
        mask_sky=False,
        mask_black_bg=False,
        mask_white_bg=False,
        target_dir=str(sample_target_dir),
        prediction_mode=prediction_mode,
    )
    glbscene.export(file_obj=str(glb_path))

    return (str(npz_path) if save_npz else None), str(glb_path)


def _to_extr44(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    if ext.ndim != 3:
        raise ValueError(f'extrinsic ndim error: {ext.shape}')
    if ext.shape[1:] == (4, 4):
        return ext
    if ext.shape[1:] == (3, 4):
        out = np.tile(np.eye(4, dtype=np.float32), (ext.shape[0], 1, 1))
        out[:, :3, :4] = ext
        return out
    raise ValueError(f'unsupported extrinsic shape: {ext.shape}')


def _from_extr44(ext44: np.ndarray, like_extr: np.ndarray) -> np.ndarray:
    like = np.asarray(like_extr)
    if like.shape[1:] == (3, 4):
        return ext44[:, :3, :4].astype(np.float32)
    if like.shape[1:] == (4, 4):
        return ext44.astype(np.float32)
    raise ValueError(f'unsupported like_extr shape: {like.shape}')


def _rotation_align_vec_to_vec(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    v = np.cross(a, b)
    c = float(np.dot(a, b))
    s = float(np.linalg.norm(v))
    if s < 1e-8:
        if c > 0:
            return np.eye(3, dtype=np.float32)
        axis = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        if abs(a[0]) > 0.9:
            axis = np.array([0.0, 1.0, 0.0], dtype=np.float32)
        v = np.cross(a, axis)
        v = v / (np.linalg.norm(v) + 1e-8)
        K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]], dtype=np.float32)
        return (np.eye(3, dtype=np.float32) + 2 * (K @ K)).astype(np.float32)
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]], dtype=np.float32)
    R = np.eye(3, dtype=np.float32) + K + K @ K * ((1 - c) / (s * s + 1e-8))
    return R.astype(np.float32)


def _apply_similarity_to_predictions(predictions: dict, s: float, R: np.ndarray, t: np.ndarray) -> dict:
    wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
    pts = wp.reshape(-1, 3)
    pts_new = (s * (pts @ R.T) + t[None, :]).astype(np.float32)
    predictions['world_points_from_depth'] = pts_new.reshape(wp.shape)

    ext_old = np.asarray(predictions['extrinsic'], dtype=np.float32)
    ext44 = _to_extr44(ext_old)
    c2w = np.linalg.inv(ext44)
    c2w_new = c2w.copy()
    c2w_new[:, :3, :3] = np.einsum('ij,njk->nik', R, c2w[:, :3, :3])
    c2w_new[:, :3, 3] = s * np.einsum('ij,nj->ni', R, c2w[:, :3, 3]) + t[None, :]
    ext_new44 = np.linalg.inv(c2w_new).astype(np.float32)
    predictions['extrinsic'] = _from_extr44(ext_new44, ext_old)
    return predictions


def apply_global_rigid_similarity(
    predictions: dict,
    mode: str = 'rigid',
    scale: float = 1.0,
    image_path: Path = None,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
) -> dict:
    mode = (mode or 'none').lower()
    if mode == 'none':
        return predictions
    if 'world_points_from_depth' not in predictions or 'extrinsic' not in predictions:
        return predictions

    src = predictions.get('camera_source', '')
    if src in ('camconf_exact', 'camconf_exact_dense'):
        if image_path is None or camconf_dir is None:
            return predictions
        H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
        file_centers, _ = _load_file_camera_centers_for_image(
            image_path=image_path,
            camconf_dir=camconf_dir,
            angle_order=camconf_angle_order,
            H=H,
            W=W,
        )
        wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
        pts = wp.reshape(-1, 3)
        pts_center = pts.mean(axis=0)
        tgt_xy = file_centers[:, :2].mean(axis=0)
        target_center = np.array([tgt_xy[0], tgt_xy[1], 0.0], dtype=np.float32)

        if mode == 'rigid':
            s = 1.0
        elif mode == 'similarity':
            s = float(scale) if float(scale) > 0 else 1.0
        else:
            pred_span = _robust_span_diag(pts, q_low=5.0, q_high=95.0)
            file_span = _robust_span_diag(file_centers, q_low=5.0, q_high=95.0)
            if camera_fit_match_range and pred_span > 1e-6 and file_span > 1e-6:
                raw_s = file_span / pred_span
                mixed_s = 1.0 + float(camera_fit_range_gain) * (raw_s - 1.0)
                s = float(np.clip(mixed_s, camera_fit_scale_min, camera_fit_scale_max))
            else:
                s = 1.0

        t = (target_center - s * pts_center).astype(np.float32)
        predictions = _apply_similarity_to_points_only(predictions, s=s, R=np.eye(3, dtype=np.float32), t=t)
        print(f'[EXACT_FIX] post-align points-only | mode={mode}, scale={s:.4f}, camera_locked=True')
        return predictions

    if mode in ('rigid', 'similarity'):
        wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
        pts = wp.reshape(-1, 3)
        center = pts.mean(axis=0)
        X = pts - center[None, :]
        _, _, vh = np.linalg.svd(X, full_matrices=False)
        normal = vh[-1]
        if normal[2] < 0:
            normal = -normal
        Rg = _rotation_align_vec_to_vec(normal.astype(np.float32), np.array([0.0, 0.0, 1.0], dtype=np.float32))
        s = 1.0 if mode == 'rigid' else float(scale)
        if s <= 0:
            s = 1.0
        t = (-s * (Rg @ center)).astype(np.float32)
        predictions = _apply_similarity_to_predictions(predictions, s=s, R=Rg, t=t)
        print(f'[post-align] mode={mode}, scale={s:.4f} (global transform applied to points+cameras)')
        return predictions

    if mode == 'camera_fit':
        if image_path is None or camconf_dir is None:
            print('[post-align] camera_fit missing image_path/camconf_dir, fallback to rigid')
            return apply_global_rigid_similarity(predictions, mode='rigid', scale=1.0)
        H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
        file_centers, valid_angles = _load_file_camera_centers_for_image(
            image_path=image_path,
            camconf_dir=camconf_dir,
            angle_order=camconf_angle_order,
            H=H,
            W=W,
        )
        pred_centers = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))
        n = min(len(pred_centers), len(file_centers))
        src_cent = pred_centers[:n].astype(np.float32)
        dst_cent = file_centers[:n].astype(np.float32)
        s_cam, R_cam, t_cam = _estimate_similarity_umeyama(src_cent, dst_cent, with_scale=True)
        predictions = _apply_similarity_to_predictions(predictions, s=s_cam, R=R_cam, t=t_cam)

        wp2 = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
        pts2 = wp2.reshape(-1, 3)
        pts_center = pts2.mean(axis=0)
        tgt_xy = dst_cent[:, :2].mean(axis=0)
        target_center = np.array([tgt_xy[0], tgt_xy[1], 0.0], dtype=np.float32)
        delta = (target_center - pts_center).astype(np.float32)
        predictions = _apply_similarity_to_predictions(
            predictions, s=1.0, R=np.eye(3, dtype=np.float32), t=delta
        )

        range_scale_applied = 1.0
        file_span = _robust_span_diag(dst_cent, q_low=5.0, q_high=95.0)
        pred_span_before = _robust_span_diag(
            np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3),
            q_low=5.0,
            q_high=95.0,
        )
        if camera_fit_match_range and pred_span_before > 1e-6 and file_span > 1e-6:
            raw_s = file_span / pred_span_before
            mixed_s = 1.0 + float(camera_fit_range_gain) * (raw_s - 1.0)
            range_scale_applied = float(np.clip(mixed_s, camera_fit_scale_min, camera_fit_scale_max))
            predictions = _apply_similarity_to_predictions(
                predictions, s=range_scale_applied, R=np.eye(3, dtype=np.float32), t=np.zeros(3, dtype=np.float32)
            )

        pred_after = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))[:n]
        rms_before = float(np.sqrt(np.mean(np.sum((src_cent - dst_cent) ** 2, axis=1))))
        rms_after = float(np.sqrt(np.mean(np.sum((pred_after - dst_cent) ** 2, axis=1))))
        print(
            f'[post-align] mode=camera_fit, used_angles={valid_angles[:n]}, '
            f'cam_scale={s_cam:.4f}, range_scale={range_scale_applied:.4f}, '
            f'range_apply_to_camera={camera_fit_range_apply_to_camera}, '
            f'rms_before={rms_before:.4f}, rms_after={rms_after:.4f}'
        )
        print(
            f'[post-align] span(file_cam)={file_span:.4f}, '
            f'span(pred_pts_before)={pred_span_before:.4f}, span(pred_pts_after)={pred_span_after:.4f}'
        )
        return predictions

    print(f'[post-align] unknown mode={mode}, skip')
    return predictions

## 8) 重建与批量执行

In [8]:
def reconstruct_one_image(
    image_path: Path,
    output_dir: Path,
    conf_thres: float,
    prediction_mode: str,
    force_resize: bool,
    save_npz: bool,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
    post_align_mode: str = 'none',
    post_align_scale: float = 1.0,
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
):
    img_rgb = read_single_image_512(str(image_path), force_resize=force_resize)
    sample_target_dir = Path(tempfile.mkdtemp(prefix=f'vggt_{image_path.stem}_'))
    _ = split_512_to_4x256_and_save(img_rgb, sample_target_dir)

    predictions = run_model(
        target_dir=sample_target_dir,
        model=model,
        image_path=image_path,
        use_camconf_camera=use_camconf_camera,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        strict_camconf=strict_camconf,
        invert_camera_direction=invert_camera_direction,
    )

    predictions = apply_global_rigid_similarity(
        predictions,
        mode=post_align_mode,
        scale=post_align_scale,
        image_path=image_path,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        camera_fit_match_range=camera_fit_match_range,
        camera_fit_range_gain=camera_fit_range_gain,
        camera_fit_scale_min=camera_fit_scale_min,
        camera_fit_scale_max=camera_fit_scale_max,
        camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
    )

    npz_path, glb_path = export_npz_and_glb(
        sample_target_dir=sample_target_dir,
        predictions=predictions,
        output_dir=output_dir,
        output_stem=image_path.stem,
        conf_thres=conf_thres,
        prediction_mode=prediction_mode,
        save_npz=save_npz,
    )

    return {
        'input_image': str(image_path),
        'target_dir': str(sample_target_dir),
        'glb_path': glb_path,
        'npz_path': npz_path,
        'camera_source': predictions.get('camera_source', 'prediction'),
    }


def reconstruct_from_input_path(
    input_path: str,
    output_path: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    force_resize: bool = True,
    save_npz: bool = True,
    valid_exts=None,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
    post_align_mode: str = 'none',
    post_align_scale: float = 1.0,
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
):
    t0 = time.time()
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    images = collect_input_images(input_path, valid_exts=valid_exts)
    print(f'[Start] total images: {len(images)}')

    results = []
    for i, img_path in enumerate(images, 1):
        print(f'[{i}/{len(images)}] processing: {img_path.name}')
        item = reconstruct_one_image(
            image_path=img_path,
            output_dir=output_dir,
            conf_thres=conf_thres,
            prediction_mode=prediction_mode,
            force_resize=force_resize,
            save_npz=save_npz,
            use_camconf_camera=use_camconf_camera,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            strict_camconf=strict_camconf,
            invert_camera_direction=invert_camera_direction,
            post_align_mode=post_align_mode,
            post_align_scale=post_align_scale,
            camera_fit_match_range=camera_fit_match_range,
            camera_fit_range_gain=camera_fit_range_gain,
            camera_fit_scale_min=camera_fit_scale_min,
            camera_fit_scale_max=camera_fit_scale_max,
            camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
        )
        results.append(item)

    elapsed = time.time() - t0
    print(f'[Done] {len(results)} image(s), elapsed: {elapsed:.2f}s')
    print('output_dir =', output_dir)
    return results

## 9) 批量重建 / 导出 GLB

In [9]:
results = reconstruct_from_input_path(
    input_path=input_path,
    output_path=output_path,
    conf_thres=conf_thres,
    prediction_mode=prediction_mode,
    force_resize=force_resize,
    save_npz=save_npz,
    valid_exts=valid_exts,
    use_camconf_camera=use_camconf_camera,
    camconf_dir=camconf_dir,
    camconf_angle_order=camconf_angle_order,
    strict_camconf=strict_camconf,
    invert_camera_direction=invert_camera_direction,
    post_align_mode=post_align_mode,
    post_align_scale=post_align_scale,
    camera_fit_match_range=camera_fit_match_range,
    camera_fit_range_gain=camera_fit_range_gain,
    camera_fit_scale_min=camera_fit_scale_min,
    camera_fit_scale_max=camera_fit_scale_max,
    camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
)

print('\n生成结果（前10条）:')
for row in results[:10]:
    print(
        f"- {Path(row['input_image']).name} -> {Path(row['glb_path']).name} "
        f"(camera={row.get('camera_source', 'prediction')})"
    )

results[:3]

[Start] total images: 9
[1/9] processing: num123_-800_100_0_0007.png
[EXACT_FIX] num123_-800_100_0_0007.png | mode=camconf_exact_dense | similarity_scale=238.2705 | points_kept@p50.0=536649
[EXACT_FIX] post-align points-only | mode=camera_fit, scale=1.7080, camera_locked=True
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[2/9] processing: num123_-800_150_0_0008.png
[EXACT_FIX] num123_-800_150_0_0008.png | mode=camconf_exact_dense | similarity_scale=202.7203 | points_kept@p50.0=536648
[EXACT_FIX] post-align points-only | mode=camera_fit, scale=1.9034, camera_locked=True
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[3/9] processing: num123_-800_200_0_0009.png
[EXACT_FIX] num123_-800_200_0_0009.png | mode=camconf_exact_dense | similarity_scale=238.0617 | points_kept@p50.0=536648
[EXACT_FIX] post-align points-only | mode=camera_fit, scale=1.5705, camera_locked=True
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[4/9] process

[{'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_100_0_0007.png',
  'target_dir': '/tmp/vggt_num123_-800_100_0_0007_lt005eks',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale3.1/num123_-800_100_0_0007.glb',
  'npz_path': None,
  'camera_source': 'camconf_exact_dense'},
 {'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_150_0_0008.png',
  'target_dir': '/tmp/vggt_num123_-800_150_0_0008_dcb5dzho',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale3.1/num123_-800_150_0_0008.glb',
  'npz_path': None,
  'camera_source': 'camconf_exact_dense'},
 {'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_200_0_0009.png',
  'target_dir': '/tmp/vggt_num123_-800_200_0_0009_z2tluwls',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale3.1/num123_-800_200_0_0009.glb',
  'npz_path': None,
  'camera_source': 'camconf_exact_dense'}]